# U.S. Memo Country Source Flags

Author: Noah Green CPA CFE

Cross-Border Diligence Atlas lab notebook. This synthetic/redacted training output turns public-source lead shapes into diligence memo flags for a U.S.-buyer-facing memo.

Boundaries:
- No jurisdiction-level score.
- No legal conclusion.
- No client names, target names, personal identifiers, or private data.
- Country pilots are source-shape examples only; jurisdiction is not treated as a proxy for risk.

## What this proves / what it cannot prove

What this proves:
- Registry-identifier output, exact-match screening output, and official-notice timeline output can be translated into owner questions, source-to-verify instructions, and escalation lanes.
- A memo flag can preserve uncertainty while making the next diligence step clear.

What it cannot prove:
- True entity status, ownership, sanctions/procurement status, litigation posture, license status, or legal effect.
- Country risk, wrongdoing, control, authority to sign, or whether a transaction should proceed.
- Whether counsel has completed analysis. Local/specialist counsel escalation is required whenever legal effect, sanctions, licensing, public procurement, litigation, insolvency, or ownership rights matter.

In [1]:
from pathlib import Path
import csv
from pprint import pprint


synthetic_source_leads = [
    {
        "sample_id": "SYN-US-MEMO-001",
        "source_shape": "registry_identifier_output",
        "lead_flag": "identifier_format_or_extract_gap",
        "synthetic_trigger": "identifier is missing, malformed, or not tied to a current official extract",
        "match_type": "identifier_quality",
        "source_name": "Official entity registry or equivalent corporate register",
        "limitation": "Identifier quality supports routing only. It does not prove current entity status.",
    },
    {
        "sample_id": "SYN-US-MEMO-002",
        "source_shape": "registry_identifier_output",
        "lead_flag": "authority_and_status_gap",
        "synthetic_trigger": "normalized identifier exists but current standing, directors, or authority fields are not verified",
        "match_type": "registry_extract_needed",
        "source_name": "Official entity registry, current extract, and filing history",
        "limitation": "A valid-looking identifier does not prove good standing, control, or authority to bind.",
    },
    {
        "sample_id": "SYN-US-MEMO-003",
        "source_shape": "exact_match_screening_output",
        "lead_flag": "official_restriction_exact_identifier_lead",
        "synthetic_trigger": "exact identifier match appears in a sanctions, debarment, procurement, or regulator-style list",
        "match_type": "exact_identifier",
        "source_name": "Official sanctions, procurement exclusion, enforcement, or regulator list",
        "limitation": "An exact identifier lead is a memo flag, not a finding. Read the official record and test scope, date, and party coverage.",
    },
    {
        "sample_id": "SYN-US-MEMO-004",
        "source_shape": "exact_match_screening_output",
        "lead_flag": "name_only_public_record_ambiguity",
        "synthetic_trigger": "normalized-name match appears without an exact identifier or sufficient disambiguating fields",
        "match_type": "exact_normalized_name",
        "source_name": "Official list plus registry records that can disambiguate name, identifier, location, and affiliates",
        "limitation": "Name-only matches are ambiguity flags. They cannot identify the party without corroborating source fields.",
    },
    {
        "sample_id": "SYN-US-MEMO-005",
        "source_shape": "official_notice_timeline_output",
        "lead_flag": "notice_timeline_legal_effect_gap",
        "synthetic_trigger": "official-notice events appear in sequence, but legal effect and current status are unclear",
        "match_type": "timeline_sequence",
        "source_name": "Official gazette, court docket, regulator notice, or filing bulletin",
        "limitation": "A notice timeline shows that events were published. It does not interpret their legal effect.",
    },
    {
        "sample_id": "SYN-US-MEMO-006",
        "source_shape": "official_notice_timeline_output",
        "lead_flag": "recent_event_timing_gap",
        "synthetic_trigger": "recent notice appears near a diligence date, signing milestone, financing step, or operating-license review",
        "match_type": "event_timing",
        "source_name": "Official notice text and any linked court, regulator, licensing, or registry file",
        "limitation": "Timing proximity is a question prompt only. It does not establish materiality or transaction impact.",
    },
]

source_lead_summary = {
    "synthetic_source_leads": len(synthetic_source_leads),
    "source_shapes": sorted({lead["source_shape"] for lead in synthetic_source_leads}),
}
source_lead_summary

{'synthetic_source_leads': 6,
 'source_shapes': ['exact_match_screening_output',
  'official_notice_timeline_output',
  'registry_identifier_output']}

In [2]:
FLAG_TEMPLATES = {
    "identifier_format_or_extract_gap": {
        "memo_flag": "Registry identifier or extract gap",
        "owner_question": "Do we have the authoritative legal name, identifier, current registry extract, and filing date from the official source?",
        "source_to_verify": "Official entity registry or equivalent corporate register",
        "escalation_lane": "Local counsel or corporate-registry specialist",
        "what_this_proves": "The memo team has a structured reason to request the official registry extract.",
        "what_it_cannot_prove": "It cannot prove existence, good standing, ownership, authority, or legal capacity.",
    },
    "authority_and_status_gap": {
        "memo_flag": "Authority, standing, or filing-status gap",
        "owner_question": "Which current source proves status, authorized signers, directors, beneficial ownership hooks, and any filed changes?",
        "source_to_verify": "Current registry extract, filing history, constitutional documents, and local status certificate where available",
        "escalation_lane": "Local counsel or corporate governance specialist",
        "what_this_proves": "The memo team can separate identifier normalization from status and authority verification.",
        "what_it_cannot_prove": "It cannot prove who controls the entity or who can bind it without authoritative documents.",
    },
    "official_restriction_exact_identifier_lead": {
        "memo_flag": "Exact-identifier public-restriction lead",
        "owner_question": "Is the exact-identifier lead current, in scope for this party, and tied to an applicable restriction, exclusion, or enforcement record?",
        "source_to_verify": "Official sanctions, procurement exclusion, enforcement, regulator, or licensing list",
        "escalation_lane": "Sanctions, export-controls, anti-corruption, or public-procurement specialist counsel",
        "what_this_proves": "The memo team has a source-specific lead that requires official-record review before reliance.",
        "what_it_cannot_prove": "It cannot prove violation, current restriction, transaction prohibition, or culpability.",
    },
    "name_only_public_record_ambiguity": {
        "memo_flag": "Name-only public-record ambiguity",
        "owner_question": "What identifier, address, officer, affiliate, date, or source field disambiguates the name-only hit?",
        "source_to_verify": "Official list record, registry record, and corroborating source fields used for disambiguation",
        "escalation_lane": "Diligence lead plus local/specialist counsel if the ambiguity could affect a memo conclusion",
        "what_this_proves": "The memo team has a possible ambiguity that should not be overread.",
        "what_it_cannot_prove": "It cannot prove the record belongs to the same party without corroborating identifiers.",
    },
    "notice_timeline_legal_effect_gap": {
        "memo_flag": "Official-notice timeline needs legal-effect review",
        "owner_question": "What did each notice actually do, when did it take effect, and does a later filing modify or cure the issue?",
        "source_to_verify": "Official gazette, court docket, regulator notice, registry filing, or linked proceeding record",
        "escalation_lane": "Local counsel or litigation, insolvency, regulatory, or corporate specialist",
        "what_this_proves": "The memo team can build a chronological source trail for counsel and deal owners.",
        "what_it_cannot_prove": "It cannot interpret legal effect, current status, or materiality without source review and counsel input.",
    },
    "recent_event_timing_gap": {
        "memo_flag": "Recent public-source event timing question",
        "owner_question": "Does the recent event affect signing authority, licenses, financing conditions, closing deliverables, or post-closing covenants?",
        "source_to_verify": "Official notice text and linked registry, court, regulator, licensing, or procurement file",
        "escalation_lane": "Local counsel, specialist counsel, and deal lead for timing/materiality triage",
        "what_this_proves": "The memo team has a dated event that should be placed against the deal timeline.",
        "what_it_cannot_prove": "It cannot establish materiality, breach, violation, or transaction impact by timing alone.",
    },
}

In [3]:
def build_memo_flags(source_leads, templates):
    rows = []
    for lead in source_leads:
        template = templates[lead["lead_flag"]]
        rows.append(
            {
                "sample_id": lead["sample_id"],
                "source_shape": lead["source_shape"],
                "memo_flag": template["memo_flag"],
                "synthetic_trigger": lead["synthetic_trigger"],
                "owner_question": template["owner_question"],
                "source_to_verify": template["source_to_verify"],
                "escalation_lane": template["escalation_lane"],
                "what_this_proves": template["what_this_proves"],
                "what_it_cannot_prove": template["what_it_cannot_prove"],
                "match_type": lead["match_type"],
                "source_name": lead["source_name"],
                "limitation": lead["limitation"],
                "redaction_status": "synthetic_redacted_no_private_data",
            }
        )
    return rows


memo_flags = build_memo_flags(synthetic_source_leads, FLAG_TEMPLATES)
pprint(memo_flags)

[{'escalation_lane': 'Local counsel or corporate-registry specialist',
  'limitation': 'Identifier quality supports routing only. It does not prove '
                'current entity status.',
  'match_type': 'identifier_quality',
  'memo_flag': 'Registry identifier or extract gap',
  'owner_question': 'Do we have the authoritative legal name, identifier, '
                    'current registry extract, and filing date from the '
                    'official source?',
  'redaction_status': 'synthetic_redacted_no_private_data',
  'sample_id': 'SYN-US-MEMO-001',
  'source_name': 'Official entity registry or equivalent corporate register',
  'source_shape': 'registry_identifier_output',
  'source_to_verify': 'Official entity registry or equivalent corporate '
                      'register',
  'synthetic_trigger': 'identifier is missing, malformed, or not tied to a '
                       'current official extract',
  'what_it_cannot_prove': 'It cannot prove existence, good standing, '


In [4]:
def atlas_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if candidate.name in {"cross-border-diligence-atlas", "cross_border_diligence_atlas"} and (candidate / "data").is_dir():
            return candidate
        nested = candidate / "SPP/articles/2026-06-14_cross_border_open_source_diligence_atlas/lab/cross-border-diligence-atlas"
        if nested.is_dir():
            return nested
    raise RuntimeError("Could not locate cross-border-diligence-atlas root")


FIELDNAMES = [
    "sample_id",
    "source_shape",
    "memo_flag",
    "synthetic_trigger",
    "owner_question",
    "source_to_verify",
    "escalation_lane",
    "what_this_proves",
    "what_it_cannot_prove",
    "match_type",
    "source_name",
    "limitation",
    "redaction_status",
]

OUTPUT_PATH = atlas_root() / "data/redacted_outputs/US_memo_country_risk_flags.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=FIELDNAMES, lineterminator="\n")
    writer.writeheader()
    writer.writerows(memo_flags)

print(f"Wrote {len(memo_flags)} synthetic memo flags to {OUTPUT_PATH}")

Wrote 6 synthetic memo flags to /Users/noah.r.green/dd-tech-lab-companions/cross_border_diligence_atlas/data/redacted_outputs/US_memo_country_risk_flags.csv


## Memo usage notes

- Use the exported rows as diligence memo prompts, not findings.
- Keep real names, private identifiers, and client workpapers outside this public lab unless publication clearance has been recorded.
- Replace synthetic samples only with publication-cleared source excerpts and verified source citations.
- Escalate to local/specialist counsel before stating legal effect, restriction scope, authority, standing, licensing impact, public-procurement effect, sanctions impact, or transaction materiality.